In [6]:
# 라이브러리 실행 확인용
import pandas as pd

In [ ]:
# 팀포지션별 승패별 평균 골드,딜량,시야,팀내 비중 
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

# 3. 서렌 가능 시간인 게임 시작 후 15분(900초) 이상 진행된 매치만
cond_duration = df_valid['gameDuration'] >= 900

# 4. 끝까지 정상적으로 완료된 매치(GameComplete)만
cond_result = df_valid['endOfGameResult'] == 'GameComplete'

# 두 조건을 모두 만족하는 데이터만 덮어쓰기
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

print(f"필터링 전 데이터 수: {len(df_valid)}행")
print(f"필터링 후 데이터 수: {len(df_filtered)}행\n")

# 5. 분석할 주요 지표 설정 (골드, 딜량, 시야, 팀내 딜 비중)
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
if 'ch_teamDamagePercentage' in df_filtered.columns:
    metrics.append('ch_teamDamagePercentage')

# 6. 승패 및 포지션별 집계 (평균, 중앙값, 표준편차)
agg_funcs = ['mean', 'median', 'std']
grouped_stats = df_filtered.groupby(['teamPosition', 'win'])[metrics].agg(agg_funcs)

# 멀티인덱스로 묶인 컬럼명을 하나로 평탄화
grouped_stats.columns = [f"{col}_{func}" for col, func in grouped_stats.columns]
grouped_stats = grouped_stats.reset_index()

# 7. 보기 좋게 데이터 정제 (한글화 및 순서 정렬)
grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})

# 포지션 정렬 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])

# 컬럼명 직관적으로 변경하는 함수
def rename_columns(col_name):
    name = col_name
    name = name.replace('goldEarned', '골드').replace('totalDamageDealtToChampions', '딜량')
    name = name.replace('visionScore', '시야').replace('ch_teamDamagePercentage', '팀내_딜비중(%)')
    name = name.replace('_mean', '_평균').replace('_median', '_중앙값').replace('_std', '_표준편차')
    name = name.replace('teamPosition', '포지션').replace('win', '승패')
    return name

grouped_stats.rename(columns=lambda x: rename_columns(x), inplace=True)

# 소수점 반올림 (가독성을 위해)
numeric_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
grouped_stats[numeric_cols] = grouped_stats[numeric_cols].round(2)

# 8. 결과 확인
print("조건이 적용된 포지션 및 승패별 분포/구조/성과 지표:")
display(grouped_stats)

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv"
grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n필터링이 적용된 분석용 데이터가 성공적으로 저장되었습니다:\n{output_path}")

필터링 전 데이터 수: 173959행
필터링 후 데이터 수: 168919행

조건이 적용된 포지션 및 승패별 분포/구조/성과 지표:


,포지션,승패,골드_평균,골드_중앙값,골드_표준편차,딜량_평균,딜량_중앙값,딜량_표준편차,시야_평균,시야_중앙값,시야_표준편차,팀내_딜비중(%)_평균,팀내_딜비중(%)_중앙값,팀내_딜비중(%)_표준편차
6,TOP,패배,10749.86,10541.5,3733.26,21691.56,19534.5,11981.89,17.67,16.0,9.65,0.23,0.22,0.07
7,TOP,승리,12553.81,12468.0,3621.76,25826.54,23981.0,12642.45,18.83,17.0,9.83,0.23,0.22,0.07
2,JUNGLE,패배,11219.93,10993.0,3652.34,17870.50,15849.5,10804.72,25.70,24.0,12.18,0.18,0.18,0.06
3,JUNGLE,승리,13093.40,13084.0,3696.23,21593.91,19773.0,11867.02,27.94,26.0,12.40,0.18,0.18,0.06
4,MIDDLE,패배,10874.82,10699.0,3652.32,21715.08,19783.0,11977.16,19.24,18.0,10.11,0.23,0.22,0.06
5,MIDDLE,승리,12486.20,12411.0,3523.81,25958.58,24283.0,12583.34,20.91,20.0,10.16,0.23,0.22,0.07
0,BOTTOM,패배,11769.77,11500.0,4152.69,21669.64,19097.0,13145.25,17.27,16.0,9.35,0.22,0.22,0.07
1,BOTTOM,승리,13848.60,13722.0,4103.68,27424.08,25047.0,14410.20,18.67,17.0,9.58,0.24,0.23,0.08
8,UTILITY,패배,8144.07,7942.0,2600.45,12788.86,10446.5,8761.82,62.04,60.0,29.85,0.13,0.12,0.06
9,UTILITY,승리,9271.13,9185.0,2539.29,14175.40,11774.0,9098.40,66.70,65.0,29.00,0.12,0.11,0.06



필터링이 적용된 분석용 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv


In [ ]:
# 챔피언별 팀라인별 매치수 픽률/승률/평균(골드/딜/시야점수)
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

# 3. 필터링 조건 적용
# 조건 1: 서렌 가능 시간인 게임 시작 후 15분(900초) 이상 진행된 매치만
cond_duration = df_valid['gameDuration'] >= 900

# 조건 2: 끝까지 정상적으로 완료된 매치(GameComplete)만
cond_result = df_valid['endOfGameResult'] == 'GameComplete'

# 조건을 모두 만족하는 정상 데이터만 추출
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

print(f"필터링 전 전체 데이터 수: {len(df_valid)}행")
print(f"조건 적용 후 데이터 수: {len(df_filtered)}행\n")

# 4. [기본 지표 계산] 매치 수, 픽률, 승률
match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'])
pick_rate = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100
win_rate = pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100

# 5. [심화 지표 계산] 골드, 딜량, 시야 점수 평균
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
grouped_avg = df_filtered.groupby(['championName', 'teamPosition'])[metrics].mean()
pivoted_avg = grouped_avg.unstack(level='teamPosition')

# 6. 열 순서 지정 및 결측치 처리
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

# 7. 모든 지표를 하나의 데이터프레임으로 병합
result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    # 7-1. 기본 지표 추가
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)
    
    # 7-2. 심화 지표(골드, 딜, 시야) 추가
    if ('goldEarned', pos) in pivoted_avg.columns:
        result_df[f'{pos}_골드평균'] = pivoted_avg[('goldEarned', pos)].fillna(0).round(1)
        result_df[f'{pos}_딜평균'] = pivoted_avg[('totalDamageDealtToChampions', pos)].fillna(0).round(1)
        result_df[f'{pos}_시야평균'] = pivoted_avg[('visionScore', pos)].fillna(0).round(1)
    else:
        result_df[f'{pos}_골드평균'] = 0.0
        result_df[f'{pos}_딜평균'] = 0.0
        result_df[f'{pos}_시야평균'] = 0.0

# 8. 결과 확인 (상위 15개 챔피언)
print("챔피언(행) x 포지션별 종합 지표 (조건 적용 완료):")
display(result_df.head(15))

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n정상 조건이 적용된 통합 마스터 데이터가 성공적으로 저장되었습니다:\n{output_path}")

필터링 전 전체 데이터 수: 173959행
조건 적용 후 데이터 수: 168919행

챔피언(행) x 포지션별 종합 지표 (조건 적용 완료):


,TOP_매치수,TOP_픽률(%),TOP_승률(%),TOP_골드평균,TOP_딜평균,TOP_시야평균,JUNGLE_매치수,JUNGLE_픽률(%),JUNGLE_승률(%),JUNGLE_골드평균,...,BOTTOM_승률(%),BOTTOM_골드평균,BOTTOM_딜평균,BOTTOM_시야평균,UTILITY_매치수,UTILITY_픽률(%),UTILITY_승률(%),UTILITY_골드평균,UTILITY_딜평균,UTILITY_시야평균
championName,,,,,,,,,,,,,,,,,,,,,
Aatrox,1372,81.72,48.69,11203.5,23538.1,17.8,240,14.29,48.75,11889.7,...,100.00,8050.0,13482.0,8.5,11,0.66,27.27,8342.3,14749.1,44.3
Ahri,20,1.20,60.00,12563.6,27640.2,25.6,0,0.00,0.00,0.0,...,50.00,11091.0,17072.5,25.5,38,2.27,23.68,8786.4,16695.4,54.7
Akali,385,17.41,51.69,11702.8,26015.5,20.9,0,0.00,0.00,0.0,...,0.00,11548.2,21125.2,23.2,4,0.18,25.00,10275.5,21716.8,45.0
Akshan,11,4.45,45.45,14114.8,27836.4,23.1,0,0.00,0.00,0.0,...,12.50,9915.9,17909.5,13.1,6,2.43,50.00,9879.3,17615.0,38.2
Alistar,20,2.30,30.00,9067.0,16714.6,16.4,3,0.35,33.33,9475.3,...,0.00,11895.3,17641.7,15.0,831,95.74,50.66,7929.3,8533.4,63.6
Ambessa,934,79.76,47.86,11201.1,24313.1,18.0,171,14.60,50.88,12136.0,...,0.00,10064.2,19227.2,14.2,2,0.17,100.00,11450.5,28465.5,35.5
Amumu,3,0.53,33.33,9952.0,17724.3,13.7,445,78.90,55.28,11075.7,...,0.00,0.0,0.0,0.0,116,20.57,56.03,8685.2,12526.2,58.4
Anivia,33,7.69,45.45,10932.7,21632.5,21.5,1,0.23,0.00,3689.0,...,50.00,9641.7,11858.0,15.0,93,21.68,46.24,9264.3,15853.4,65.2
Annie,22,6.65,40.91,11184.8,26106.2,22.0,0,0.00,0.00,0.0,...,50.00,9272.8,16337.5,15.3,42,12.69,52.38,9539.4,20193.2,68.2



정상 조건이 적용된 통합 마스터 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv
